In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
BATCH_SIZE = 32
LR = 0.001
EPOCHS = 25
IMG_SIZE = 224
NUM_CLASSES = 3

In [4]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [5]:
import os

def clean_hidden(folder):
    for root, dirs, files in os.walk(folder):
        for d in dirs:
            if d.startswith('.'):
                os.rmdir(os.path.join(root, d))

clean_hidden("dataset")

In [6]:
import splitfolders

splitfolders.ratio(
    "dataset",
    output = "output_dataset",
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 1947 files [00:06, 279.00 files/s]


In [7]:
train_dataset = datasets.ImageFolder("output_dataset/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("output_dataset/val", transform=val_transforms)
test_dataset = datasets.ImageFolder("output_dataset/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['normal', 'osteopenia', 'osteoporosis']


In [8]:
# ----------------------------
# CNN Model Definition
# ----------------------------
class OsteoCNN(nn.Module):
    def __init__(self):
        super(OsteoCNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = OsteoCNN().to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
# ----------------------------
# Training Loop
# ----------------------------
def train_model():
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss:.4f}, Accuracy: {train_acc:.2f}%")

train_model()

Epoch [1/25], Loss: 309.6046, Accuracy: 49.19%
Epoch [2/25], Loss: 38.7777, Accuracy: 61.09%
Epoch [3/25], Loss: 33.5656, Accuracy: 63.44%
Epoch [4/25], Loss: 32.1893, Accuracy: 64.10%
Epoch [5/25], Loss: 31.0845, Accuracy: 66.67%
Epoch [6/25], Loss: 30.3289, Accuracy: 68.36%
Epoch [7/25], Loss: 28.8795, Accuracy: 69.82%
Epoch [8/25], Loss: 29.6110, Accuracy: 67.25%
Epoch [9/25], Loss: 29.7727, Accuracy: 66.96%
Epoch [10/25], Loss: 29.0109, Accuracy: 68.80%


In [ ]:
# ----------------------------
# Evaluation
# ----------------------------
def evaluate_model(loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

evaluate_model(test_loader)

# Save model
torch.save(model.state_dict(), "osteoporosis_cnn.pth")
print("Model saved successfully!")

In [ ]:
from collections import Counter
Counter(train_dataset.targets)